In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

# Silver sources
order_items = spark.table("ecommerce_dev.silver.order_items")
orders      = spark.table("ecommerce_dev.silver.orders")

# Gold dimensions — SCD2 dims loaded WITHOUT is_current filter (point-in-time join needed)
dim_customer = spark.table("ecommerce_dev.gold.dim_customer")
dim_seller   = spark.table("ecommerce_dev.gold.dim_seller")
dim_product  = spark.table("ecommerce_dev.gold.dim_product")
dim_date     = spark.table("ecommerce_dev.gold.dim_date")

In [0]:
spark.sql("""
  UPDATE ecommerce_dev.gold.dim_seller
  SET effective_date = TIMESTAMP('1900-01-01')
  WHERE is_current = true AND end_date IS NULL
""")

spark.sql("""
  UPDATE ecommerce_dev.gold.dim_product
  SET effective_date = TIMESTAMP('1900-01-01')
  WHERE is_current = true AND end_date IS NULL
""")

In [0]:
fact_order_items = (
    order_items
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp", "order_status"),
          "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"),
          "customer_id", "left")
    .join(
        dim_seller.select(col("seller_key"), col("seller_id").alias("_seller_id"),
                           col("effective_date").alias("seller_effective_date"),
                           col("end_date").alias("seller_end_date")),
        (col("seller_id") == col("_seller_id")) &
        (to_date("order_purchase_timestamp") >= col("seller_effective_date")) &
        (col("seller_end_date").isNull() | (to_date("order_purchase_timestamp") < col("seller_end_date"))),
        "left"
    )
    .join(
        dim_product.select(col("product_key"), col("product_id").alias("_product_id"),
                            col("effective_date").alias("product_effective_date"),
                            col("end_date").alias("product_end_date")),
        (col("product_id") == col("_product_id")) &
        (to_date("order_purchase_timestamp") >= col("product_effective_date")) &
        (col("product_end_date").isNull() | (to_date("order_purchase_timestamp") < col("product_end_date"))),
        "left"
    )
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "order_id", "order_item_id",
        "customer_key", "seller_key", "product_key", "order_date_key",
        "order_status", "price", "freight_value"
    )
)

In [0]:
total = fact_order_items.count()
nulls = fact_order_items.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in ["customer_key","seller_key","product_key","order_date_key"]]
)
print(f"Row count: {total}")
nulls.display()

Row count: 112650


customer_key,seller_key,product_key,order_date_key
0,0,0,0


In [0]:
target = DeltaTable.forName(spark, "ecommerce_dev.gold.fact_order_items")

(target.alias("t")
 .merge(
     fact_order_items.alias("s"),
     "t.order_id = s.order_id AND t.order_item_id = s.order_item_id"
 )
 .whenMatchedUpdate(set={
     "customer_key": "s.customer_key",
     "seller_key": "s.seller_key",
     "product_key": "s.product_key",
     "order_date_key": "s.order_date_key",
     "order_status": "s.order_status",
     "price": "s.price",
     "freight_value": "s.freight_value"
 })
 .whenNotMatchedInsert(values={
     "order_id": "s.order_id",
     "order_item_id": "s.order_item_id",
     "customer_key": "s.customer_key",
     "seller_key": "s.seller_key",
     "product_key": "s.product_key",
     "order_date_key": "s.order_date_key",
     "order_status": "s.order_status",
     "price": "s.price",
     "freight_value": "s.freight_value"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
  SELECT COUNT(*) AS total,
         SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer,
         SUM(CASE WHEN seller_key IS NULL THEN 1 ELSE 0 END) AS null_seller,
         SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS null_product,
         SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END) AS null_date
  FROM ecommerce_dev.gold.fact_order_items
""").show()

+------+-------------+-----------+------------+---------+
| total|null_customer|null_seller|null_product|null_date|
+------+-------------+-----------+------------+---------+
|112650|            0|          0|           0|        0|
+------+-------------+-----------+------------+---------+



In [0]:
order_payments = spark.table("ecommerce_dev.silver.order_payments")

fact_payments = (
    order_payments
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "order_id",
        "payment_sequential",
        "customer_key",
        "order_date_key",
        "payment_type",
        "payment_installments",
        "payment_value",
        "has_invalid_installments"
    )
)

In [0]:
total = fact_payments.count()
nulls = fact_payments.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in ["customer_key", "order_date_key"]]
)
print(f"Row count: {total}")
nulls.show()

Row count: 103886
+------------+--------------+
|customer_key|order_date_key|
+------------+--------------+
|           0|             0|
+------------+--------------+



In [0]:
target = DeltaTable.forName(spark, "ecommerce_dev.gold.fact_payments")

(target.alias("t")
 .merge(
     fact_payments.alias("s"),
     "t.order_id = s.order_id AND t.payment_sequential = s.payment_sequential"
 )
 .whenMatchedUpdate(set={
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "payment_type": "s.payment_type",
     "payment_installments": "s.payment_installments",
     "payment_value": "s.payment_value",
     "has_invalid_installments": "s.has_invalid_installments"
 })
 .whenNotMatchedInsert(values={
     "order_id": "s.order_id",
     "payment_sequential": "s.payment_sequential",
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "payment_type": "s.payment_type",
     "payment_installments": "s.payment_installments",
     "payment_value": "s.payment_value",
     "has_invalid_installments": "s.has_invalid_installments"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
  SELECT COUNT(*) AS total,
         SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer,
         SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END) AS null_date
  FROM ecommerce_dev.gold.fact_payments
""").show()

+------+-------------+---------+
| total|null_customer|null_date|
+------+-------------+---------+
|103886|            0|        0|
+------+-------------+---------+



In [0]:
order_reviews = spark.table("ecommerce_dev.silver.order_reviews")

fact_reviews = (
    order_reviews
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "review_id",
        "order_id",
        "customer_key",
        "order_date_key",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    )
)

In [0]:
total = fact_reviews.count()
nulls = fact_reviews.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in ["customer_key", "order_date_key"]]
)
print(f"Row count: {total}")
nulls.show()


Row count: 99224
+------------+--------------+
|customer_key|order_date_key|
+------------+--------------+
|           0|             0|
+------------+--------------+



In [0]:
target = DeltaTable.forName(spark, "ecommerce_dev.gold.fact_reviews")

(target.alias("t")
 .merge(
     fact_reviews.alias("s"),
     "t.review_id = s.review_id AND t.order_id = s.order_id"
 )
 .whenMatchedUpdate(set={
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "review_score": "s.review_score",
     "review_comment_title": "s.review_comment_title",
     "review_comment_message": "s.review_comment_message",
     "review_creation_date": "s.review_creation_date",
     "review_answer_timestamp": "s.review_answer_timestamp"
 })
 .whenNotMatchedInsert(values={
     "review_id": "s.review_id",
     "order_id": "s.order_id",
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "review_score": "s.review_score",
     "review_comment_title": "s.review_comment_title",
     "review_comment_message": "s.review_comment_message",
     "review_creation_date": "s.review_creation_date",
     "review_answer_timestamp": "s.review_answer_timestamp"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
  SELECT COUNT(*) AS total,
         SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer,
         SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END) AS null_date
  FROM ecommerce_dev.gold.fact_reviews
""").show()

+-----+-------------+---------+
|total|null_customer|null_date|
+-----+-------------+---------+
|99224|            0|        0|
+-----+-------------+---------+

